# SGP Kits Statistics Report

This notebook analyzes the normalized kit statistics extracted from `command_storage.dat`.

The main focus is kit-level performance. Player IDs are kept as an explanatory dimension so we can detect cases where a kit's aggregate kill count is heavily driven by one or a few players.

## Setup

The companion `sgp_data.py` module handles loading, validation, and derived datasets. `sgp_report.py` defines the SGP-specific figures, while `sgp_plot_components.py` contains their reusable interactive mechanics. This keeps the notebook focused on the analysis itself.


In [ ]:
from pathlib import Path

from sgp_data import load_report_data
from sgp_report import (
    ability_uses_figure,
    kill_causes_figure,
    kill_concentration_figure,
    kill_concentration_scatter_figure,
    kills_vs_ability_uses_figure,
    matchup_figure,
    player_reach_figure,
    popularity_efficiency_figure,
    report_summary_figure,
    show_player_contribution_figure,
    top_killer_exposure_figure,
    total_kills_figure,
)


report = load_report_data(Path("data"))


# Kit adoption

## Proportion of players who tried each kit

The denominator is every player appearing anywhere in the extracted kill, ability-use, or kit-exposure data.

The default view uses observed playtime directly. Each wide, faded bar is the proportion who tried the kit; the narrower solid bar inside it is the proportion who also made at least one player kill. Ability reach remains available on hover.

The **Exposure shares** mode shows playtime and completed lives as two 100% stacked bars. Both use the same kit order, sorted by playtime share, so their segment sizes can be compared directly. **Life duration** shows average observed time per completed life.

These views distinguish broad player adoption from how heavily and how repeatedly each kit was played. Per-completed-life values use the aggregate counter as their denominator, so an unfinished current life is not counted as completed.

In [ ]:
fig = player_reach_figure(report)
fig.show()


# Kills

## Total kills by kit

Switch among total kills, the same totals stacked by player contribution, kills per active hour, kills per completed life, and kills per player-caused death. The **Kills / PvP death** mode compares attributed kills made with a kit against deaths attributed to a player while using that kit; its black line marks parity at 1.0, and non-player deaths remain available on hover. Rate hovers compare the server-wide result with the median eligible player's rate. In the kills-per-life mode, the thick black line marks 1.0 player kill per completed life and the dashed line marks the kit median. Player IDs appear on hover in the stacked view; click a segment to emphasize that player across every bar, then click it again to restore all colors.

In [ ]:
fig = total_kills_figure(report)
show_player_contribution_figure(fig)


## Kill causes as offensive identity and defensive vulnerability

The **Outgoing share** mode shows how each attacking kit delivers its attributed player kills. A kit dominated by one cause has a narrow offensive identity; a mixed profile indicates that several damage mechanisms regularly finish opponents. Only deaths with a real attacking kit and victim kit are included in this mode.

The two incoming modes reverse the perspective. **Incoming deaths / hour** compares each victim kit's exposure-normalized death rate and stacks the causes responsible; **Incoming share** removes the rate magnitude so cause susceptibility can be compared directly. These modes include deaths with no player killer, but exclude deaths where the victim had no kit. Cause labels use the names stored with the statistics.

In [ ]:
fig = kill_causes_figure(report)
fig.show()


## Playtime share vs. kill efficiency

This separates popularity from exposure-normalized kill output. The vertical dashed line is the equal-share benchmark (one twelfth of kit playtime); the horizontal dashed line is the edition's overall observed kills per active hour. A point above the horizontal line produced kills faster than the server-wide rate, while its horizontal position shows whether players devoted more or less than an equal share of kit playtime to it.

This is still an FFA output measure, not a pure combat win rate: play style, map behavior, and the kinds of fights a kit takes can also affect kills per hour.

In [ ]:
fig = popularity_efficiency_figure(report)
fig.show()


## Player concentration of kills

Each bar partitions a kit's activity into the top player, players 2–3, and everyone else. Switch among kills, playtime, and completed lives. The contributor count above each bar gives the context needed to interpret a concentrated result.

In [ ]:
fig = kill_concentration_figure(report)
fig.show()


## Total kills vs. player concentration

Each point is a kit. The dashed median lines divide the plot into four descriptive quadrants, separating kill volume from how strongly the result depends on the top player. Hover also shows playtime, completed lives, kills per hour, and kills per completed life.

In [ ]:
fig = kill_concentration_scatter_figure(report)
fig.show()


## Top-killer output relative to exposure

For each kit, both coordinates describe the same player: the player with the most kills using that kit. The x-axis is that player's share of the kit's playtime and the y-axis is their share of its kills. Above the diagonal, the top killer contributed a larger share of kills than of playtime; below it, their apparent kill dominance is more than explained by how much of the kit's playtime they supplied.

This makes the concentration result easier to interpret without treating every top-player share as equally suspicious. Kits without valid playtime for their top killer are omitted.

In [ ]:
fig = top_killer_exposure_figure(report)
fig.show()


# Matchups

These views describe observed kill counts between kits. They are not win-rate estimates because the dataset does not contain the number of encounters or time played in each matchup.

## Directional kill share and raw kill matrix

The default view uses one cell per kit pair. A value above 50% means the row kit killed the column kit more often than the reverse. Hover a cell for the exact share, pair volume, and cause breakdown in both directions. Use the button to switch to the full raw kill matrix, whose cell hovers show the cause mix for that exact attacker-victim direction.

In [ ]:
fig = matchup_figure(
    report.matchup_matrix,
    report.directional_share,
    report.pair_totals,
    report.matchup_kills_by_cause,
)
fig.show()


# Ability usage

Ability-use counts show current engagement patterns. Because cooldowns differ by kit, raw uses per hour are retained as context but are no longer treated as the main cross-kit intensity comparison.

## Ability uses by kit

Switch among total uses, player-stacked contributions, cooldown-normalized use, and uses per completed life. **Cooldown-normalized use** divides the observed uses-per-hour rate by the maximum rate implied by the configured cooldown alone. For example, 25% means one activation for roughly every four cooldown lengths of playtime. It is not a combat-opportunity percentage: travel and downtime remain in the denominator, and cooldown resets can make values exceed 100%.

The hover retains raw uses per hour, cooldown, cooldown-only maximum, and the median eligible player's normalized rate. Player IDs appear in the stacked view; click a segment to emphasize that player across every bar, then click it again to restore all colors.

In [ ]:
fig = ability_uses_figure(report)
show_player_contribution_figure(fig)


# Combined analysis

## Kills vs. ability uses

Each point is a kit. Switch among aggregate totals, cooldown-normalized ability use against kills per hour, and per-completed-life rates; median lines and quadrant labels are recalculated for every mode. The cooldown-normalized mode replaces the former raw uses-per-hour comparison so short-cooldown kits are not automatically pushed to the right. The comparison does not imply that ability use causes kills.

In [ ]:
fig = kills_vs_ability_uses_figure(report.combined_totals)
fig.show()


# Summary

The final plot aligns four complementary views of every kit while keeping each metric on a meaningful scale. Kits are sorted by total kills. The kill hover now includes deaths per hour, K/D, and the non-player death share. The ability panel uses cooldown-normalized activation intensity instead of raw ability volume; total uses and raw hourly rates remain on hover and in the dedicated ability plot. The reach panel distinguishes actual play, ability use, and making a kill. Vertical guides show edition medians, with the reach guides corresponding to actual play and kill reach.

In [ ]:
fig = report_summary_figure(report)
fig.show()
